# Práctica 2 — Deep Q-Network (DQN)
**Materia:** Aprendizaje por Refuerzo  
**Maestría en Ciencias en Inteligencia Artificial — UAQ**

## Objetivo
Extender Q-Learning a espacios de estado continuos mediante una red neuronal (DQN), usando Experience Replay y una red objetivo (target network) para estabilizar el entrenamiento en CartPole-v1.

In [ ]:
!pip install gymnasium tensorflow --quiet

In [ ]:
import gymnasium as gym
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import random
from collections import deque
import matplotlib.pyplot as plt
print(f"TF {tf.__version__} | Gym {gym.__version__}")

## 1. Definición de la red neuronal Q

In [ ]:
def build_dqn(state_dim, action_dim):
    model = models.Sequential([
        layers.Dense(64, activation='relu', input_shape=(state_dim,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(action_dim, activation='linear')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    return model

## 2. Agente DQN con Experience Replay

In [ ]:
class DQNAgent:
    def __init__(self, state_dim, action_dim):
        self.action_dim = action_dim
        self.memory = deque(maxlen=10000)
        self.gamma   = 0.99
        self.epsilon = 1.0
        self.eps_min = 0.01
        self.eps_dec = 0.995
        self.batch   = 64
        self.model   = build_dqn(state_dim, action_dim)
        self.target  = build_dqn(state_dim, action_dim)
        self.target.set_weights(self.model.get_weights())

    def act(self, state):
        if random.random() < self.epsilon:
            return random.randrange(self.action_dim)
        return np.argmax(self.model.predict(state[np.newaxis], verbose=0)[0])

    def remember(self, s, a, r, s2, done):
        self.memory.append((s, a, r, s2, done))

    def replay(self):
        if len(self.memory) < self.batch: return
        batch = random.sample(self.memory, self.batch)
        S  = np.array([b[0] for b in batch])
        A  = np.array([b[1] for b in batch])
        R  = np.array([b[2] for b in batch])
        S2 = np.array([b[3] for b in batch])
        D  = np.array([b[4] for b in batch])
        targets = self.model.predict(S, verbose=0)
        Q_next  = self.target.predict(S2, verbose=0).max(axis=1)
        targets[np.arange(self.batch), A] = R + self.gamma * Q_next * (1 - D)
        self.model.fit(S, targets, verbose=0, epochs=1)
        self.epsilon = max(self.eps_min, self.epsilon * self.eps_dec)

    def update_target(self):
        self.target.set_weights(self.model.get_weights())

## 3. Entrenamiento

In [ ]:
env    = gym.make('CartPole-v1')
agent  = DQNAgent(state_dim=4, action_dim=2)
ep_recompensas = []

for ep in range(300):
    state, _ = env.reset()
    total_r = 0
    for _ in range(500):
        action = agent.act(state)
        next_state, reward, done, truncated, _ = env.step(action)
        agent.remember(state, action, reward, next_state, done or truncated)
        agent.replay()
        state = next_state
        total_r += reward
        if done or truncated: break
    if ep % 10 == 0:
        agent.update_target()
    ep_recompensas.append(total_r)
    if ep % 50 == 0:
        print(f"Ep {ep:3d} | Recompensa: {total_r:6.1f} | ε: {agent.epsilon:.3f}")

In [ ]:
ventana = 20
medias = [np.mean(ep_recompensas[max(0,i-ventana):i+1]) for i in range(len(ep_recompensas))]
plt.figure(figsize=(10,4))
plt.plot(ep_recompensas, alpha=0.4, color='steelblue')
plt.plot(medias, color='navy', linewidth=2, label=f'Media móvil ({ventana})')
plt.axhline(195, color='red', linestyle='--', label='Umbral de resolución (195)')
plt.xlabel('Episodio'); plt.ylabel('Recompensa total')
plt.title('DQN en CartPole-v1'); plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig('dqn_cartpole.png', dpi=120); plt.show()

## Conclusiones
- Experience Replay rompe la correlación temporal entre muestras consecutivas, estabilizando el entrenamiento.
- La red objetivo (target network) evita oscilaciones al fijar los valores Q de referencia durante varios pasos.
- DQN resuelve CartPole consistentemente en menos de 300 episodios con esta configuración.